# Environment Setup and Authentication

In [1]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
import transformers

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | Detected {device_count} GPUs.")
for i in range(device_count):
    print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

# Fail withoust falling back to CPU
assert device_count > 0, "No GPU detected. Session options -> Accelerator -> GPU T4 x2, then restart."

from transformers import AutoModelForCausalLM, AutoTokenizer

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("Authenticated with HF Hub.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 86.3 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.9/784.9 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 85.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | transformers 5.15.0 | Detected 2 GPUs.
  cuda:0 -> Tesla T4
  cuda:1 -> Tesla T4
Authenticated with HF Hub.


# Evaluation Metrices

In [2]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0", max_length=None):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        # Pythia-410M has max_position_embeddings = 2048 if this is not provided cuda fails
        self.max_length = max_length or getattr(model.config, "max_position_embeddings", 2048)

    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL and normalizes it into BPB, BPC, and BPW."""
        if not isinstance(text, str):
            text = str(text)

        inputs = self.tokenizer(
            text, return_tensors="pt",
            truncation=True, max_length=self.max_length,
            add_special_tokens=False          # harmonises OPT with the no-BOS models
        ).to(self.device)
        seq_len = inputs["input_ids"].shape[1]

        # seq_len == 1 gives zero scored label positions -> loss is NaN
        # NaN * 0 is still NaN, which would poison the column mean silently
        if seq_len < 2:
            return {"nll": float("nan"), "bpb": float("nan"),
                    "bpc": float("nan"), "bpw": float("nan"),
                    "n_scored": 0, "n_words": max(1, len(text.strip().split())),
                    "ppl_token": float("nan"), "ppl_word": float("nan")}

        # If truncation fired, normalise against the text the model actually
        # scored, otherwise BPB is deflated by bytes never seen by the model.
        if seq_len >= self.max_length:
            text = self.tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
        chars, bytes_count, words = self.get_lengths(text)

        # inference_mode is strictly cheaper than no_grad
        with torch.inference_mode():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            # HF CausalLM loss is the average negative log-likelihood per token
            neg_log_likelihood = outputs.loss.item() * (seq_len - 1)

        # Calculate Tokenizer-Independent Metrics
        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))

        return {
            "nll": neg_log_likelihood,
            "bpb": bpb,
            "bpc": bpc,
            "bpw": bpw,
            # Needed for corpus-level perplexity. n_scored is (seq_len - 1)
            # because the first token is context and is never predicted.
            "n_scored": seq_len - 1,
            "n_words": words,
            # Per-sequence PPLs. Token PPL is tokenizer-dependent -- do not
            # place it in a cross-model table. Word PPL == 2**bpw is not.
            "ppl_token": math.exp(neg_log_likelihood / (seq_len - 1)),
            "ppl_word": 2 ** bpw,
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


# Data Loading and Preprocessing

In [3]:
diverse_sentences_path = "/kaggle/input/datasets/uom220538d/intrinsic-evaluation-dataset/diverse_sentences.csv"
sinhala_mixed_path = "/kaggle/input/datasets/uom220538d/intrinsic-evaluation-dataset/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


# Model & Tokenizer Initialization

In [4]:
model_id = "meta-llama/Llama-3.1-8B-Instruct"
print(f"Loading {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 across both GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype=torch.float16,
    attn_implementation="sdpa",
    low_cpu_mem_usage=True,
    token=HF_TOKEN
)
model.eval()
model.config.use_cache = False   # no generation here; skips KV allocation

# Initialize our custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Context: {evaluator.max_length}")
print(f"Vocab: {len(tokenizer)}")
print(f"Device map: {getattr(model, 'hf_device_map', 'single-device')}")

# Llama-3 prepends <|begin_of_text|>; add_special_tokens=False suppresses it,
# keeping the (seq_len - 1) convention identical to the other ten runs.
probe = tokenizer("test", return_tensors="pt", add_special_tokens=False)["input_ids"][0]
print(f"BOS suppressed: {probe[0].item() != getattr(tokenizer, 'bos_token_id', None)}")

Loading meta-llama/Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded.
Precision: torch.float16 | Context: 131072
Vocab: 128256
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}
BOS suppressed: True


# Pilot Run

In [5]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

# Open CSV in append mode to protect against runtime crashes
with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    # Write standardized header
    writer.writerow([
        'item_index', 
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nwords'
    ])
    
    # Slice the first 20 rows of our loaded Kaggle dataframe
    pilot_sample = diverse_df.head(20)
    
    for counter, (index, row) in enumerate(pilot_sample.iterrows(), start=1):
        # 1. Evaluate Native Unicode
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        
        # 2. Evaluate Romanized (Singlish)
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])
        
        # Log to disk
        writer.writerow([
            index, 
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
            rom_metrics['n_scored'], rom_metrics['n_words']
        ])
        
        if counter % 5 == 0:
            print(f"Processed {counter}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")

# Display verification preview
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

# fp16 sanity gate: stop here if the forward pass produced NaN/inf.
import numpy as np
vals = preview_df.drop(columns=['item_index']).to_numpy(dtype=float)
assert np.isfinite(vals).all(), "Non-finite values in pilot output -- inspect before the full run."
print(f"\nPilot means -> Unicode BPB {preview_df['unicode_bpb'].mean():.4f} | "
      f"Romanized BPB {preview_df['romanized_bpb'].mean():.4f}")

Starting Pilot Run (First 20 items)...
Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,unicode_nscored,unicode_nwords,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw,romanized_nscored,romanized_nwords
0,0,70.702012,1.307711,3.400048,17.000240,53,6,93.426286,3.546991,3.546991,22.464273,13,6
1,1,58.317317,1.274759,3.505588,21.033526,44,4,81.176983,4.182630,4.182630,29.278408,9,4
2,2,69.935920,1.215617,3.254716,16.816034,54,6,95.891297,3.842831,3.842831,23.056983,13,6



Pilot means -> Unicode BPB 1.4265 | Romanized BPB 3.8354


# Full Intrinsic Evaluation

In [6]:
import csv
import pandas as pd
import numpy as np
from tqdm import tqdm

# Evaluate Parallel Diverse Sentences (Unicode vs Romanized)
full_diverse_output = 'llama_3_1_8b_instruct_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'unicode_nscored', 'unicode_nwords',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw',
        'romanized_nscored', 'romanized_nwords'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            uni_metrics['n_scored'], uni_metrics['n_words'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw'],
             rom_metrics['n_scored'], rom_metrics['n_words']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

# Evaluate Authentic Mixed-Script Sentences
full_mixed_output = 'llama_3_1_8b_instruct_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw', 'mixed_nscored', 'mixed_nwords'])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw'],
            mix_metrics['n_scored'], mix_metrics['n_words']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")

# Aggregate and Display Unicode vs Romanized Summary
def calculate_full_aggregates(csv_path, model_label="Llama-3.1-8B-Instruct"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Unicode (Native)': [
            df['unicode_bpb'].mean(),
            df['unicode_bpc'].mean(),
            df['unicode_bpw'].mean()
        ],
        'Romanized (Singlish)': [
            df['romanized_bpb'].mean(),
            df['romanized_bpc'].mean(),
            df['romanized_bpw'].mean()
        ]
    }

    results_df = pd.DataFrame(summary)
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)


    ppl = pd.DataFrame({
        'Perplexity': ['Token-level (tokenizer-DEPENDENT)', 'Word-level (corpus aggregate)'],
        'Unicode (Native)': [
            np.exp(df['unicode_nll'].sum() / df['unicode_nscored'].sum()),
            np.exp(df['unicode_nll'].sum() / df['unicode_nwords'].sum()),
        ],
        'Romanized (Singlish)': [
            np.exp(df['romanized_nll'].sum() / df['romanized_nscored'].sum()),
            np.exp(df['romanized_nll'].sum() / df['romanized_nwords'].sum()),
        ],
    })
    ppl['Ratio (Rom/Uni)'] = ppl['Romanized (Singlish)'] / ppl['Unicode (Native)']

    print(f"\n      {model_label} PERPLEXITY")
    print("==========================================================")
    print(ppl.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print("==========================================================")
    print("NOTE: token-level PPL is not comparable across models with "
          "different tokenizers; use the word-level row for cross-model tables.")
    display(ppl)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

Starting full parallel evaluation on 500 items...


Processing Parallel Pairs: 100%|██████████| 500/500 [01:26<00:00,  5.77it/s]


Diverse evaluation complete. Saved to llama_3_1_8b_instruct_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [00:51<00:00,  9.65it/s]


Mixed-script evaluation complete. Saved to llama_3_1_8b_instruct_mixed_intrinsic.csv
      Llama-3.1-8B-Instruct INTRINSIC EVALUATION SUMMARY
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            1.4295                3.6388              2.5455
Bits-Per-Character (BPC)            3.7795                3.6468              0.9649
     Bits-Per-Word (BPW)           20.2329               22.4627              1.1102


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),1.4295,3.6388,2.55×
1,Bits-Per-Character (BPC),3.7795,3.6468,0.96×
2,Bits-Per-Word (BPW),20.2329,22.4627,1.11×



      Llama-3.1-8B-Instruct PERPLEXITY
                       Perplexity  Unicode (Native)  Romanized (Singlish)  Ratio (Rom/Uni)
Token-level (tokenizer-DEPENDENT)              4.23              1,634.83           386.55
    Word-level (corpus aggregate)        905,514.21          4,299,416.91             4.75
NOTE: token-level PPL is not comparable across models with different tokenizers; use the word-level row for cross-model tables.


,Perplexity,Unicode (Native),Romanized (Singlish),Ratio (Rom/Uni)
0,Token-level (tokenizer-DEPENDENT),4.229320,1.634825e+03,386.545693
1,Word-level (corpus aggregate),905514.213026,4.299417e+06,4.748039


In [7]:
def calculate_mixed_aggregates(csv_path, model_label="Llama-3.1-8B-Instruct"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Mixed-Script': [
            df['mixed_bpb'].mean(),
            df['mixed_bpc'].mean(),
            df['mixed_bpw'].mean()
        ],
        'Std Dev': [
            df['mixed_bpb'].std(),
            df['mixed_bpc'].std(),
            df['mixed_bpw'].std()
        ]
    }

    results_df = pd.DataFrame(summary)

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script': '{:.4f}',
        'Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    # Word count is recoverable from bpw and nll even in older CSVs that predate
    # the n_words column: bpw = nll / (words * ln2)  =>  words = nll / (bpw * ln2)
    if 'mixed_nwords' in df.columns:
        total_words = df['mixed_nwords'].sum()
    else:
        total_words = (df['mixed_nll'] / (df['mixed_bpw'] * np.log(2))).round().sum()
        print("(word counts reconstructed from nll/bpw — column absent in this CSV)")

    rows = [{
        'Perplexity': 'Word-level (corpus aggregate)',
        'Mixed-Script': np.exp(df['mixed_nll'].sum() / total_words)
    }]

    # Token-level needs the token count, which is NOT recoverable from the
    # stored columns. Only present if the run wrote mixed_nscored.
    if 'mixed_nscored' in df.columns:
        rows.insert(0, {
            'Perplexity': 'Token-level (tokenizer-DEPENDENT)',
            'Mixed-Script': np.exp(df['mixed_nll'].sum() / df['mixed_nscored'].sum())
        })

    ppl = pd.DataFrame(rows)
    print(f"\n      {model_label} MIXED-SCRIPT PERPLEXITY")
    print("==========================================================")
    print(ppl.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print("==========================================================")
    if 'mixed_nscored' not in df.columns:
        print("NOTE: token-level PPL unavailable — mixed_nscored not in this CSV.")
    display(ppl)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)

      Llama-3.1-8B-Instruct MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script  Std Dev
     Bits-Per-Byte (BPB)        2.0389   0.5738
Bits-Per-Character (BPC)        3.8762   0.6033
     Bits-Per-Word (BPW)       23.5608   4.5139


,Metric,Mixed-Script,Std Dev
0,Bits-Per-Byte (BPB),2.0389,0.5738
1,Bits-Per-Character (BPC),3.8762,0.6033
2,Bits-Per-Word (BPW),23.5608,4.5139



      Llama-3.1-8B-Instruct MIXED-SCRIPT PERPLEXITY
                       Perplexity  Mixed-Script
Token-level (tokenizer-DEPENDENT)          8.80
    Word-level (corpus aggregate)  7,214,942.94


,Perplexity,Mixed-Script
0,Token-level (tokenizer-DEPENDENT),8.800918e+00
1,Word-level (corpus aggregate),7.214943e+06
